# Примитивный ETL pipeline, исследование API Steam

### Вводное

In [ ]:
import requests
import json
import pandas as pd

In [ ]:
domain = f"https://api.steampowered.com"
keyApi = "7296C755A9D4630856C6675624278EDD"


In [ ]:
def getSteamApi(interface, method, version):
    objectRequest = requests.get(url=domain + f"/{interface}/{method}/v{version}")
    return objectRequest


def getSteamApiPars(interface, method, version, pars: dict):
    objectRequest = requests.get(url=domain + f"/{interface}/{method}/v{version}", params=pars)
    return objectRequest


## Интерфейсы и методы

### Запрос интерфейсов

In [ ]:
# Запрос к по
apiRequest = getSteamApiPars("ISteamWebAPIUtil", "GetSupportedAPIList", "1", {"key": keyApi})
print(apiRequest.status_code)

### Список GET интерфейсов и методов

In [ ]:
interfaces = apiRequest.json()["apilist"]["interfaces"]  # список интерфейсов и их методов

interfacesGet = dict()  # get интерфейсы и методы, ключ: название интерфейса, значение: список методов

for interface in interfaces:
    interfaceName = interface["name"]
    for method in interface["methods"]:
        if "httpmethod" in method and method["httpmethod"] == "GET":
            if interfaceName in interfacesGet:
                interfacesGet[interfaceName].append(method)
            else:
                interfacesGet[interfaceName] = [method]

interfacesGetList = interfacesGet.keys()


In [ ]:
print(interfacesGetList)
print(len(interfacesGetList))

In [ ]:
interfaceName = "IStoreService"

interfaceCurrent = interfacesGet[interfaceName]

print(pd.DataFrame(interfaceCurrent))


## Extract

### Cписок всех доступных игр в Steam

In [ ]:
# https://partner.steamgames.com/doc/webapi/IStoreService#GetAppList

json_params = {
    "max_results": 50000
}

params = {
    "key": keyApi,
    "input_json": json.dumps(json_params)
}

appList = list()

while True:
    requestAppList = getSteamApiPars("IStoreService", "GetAppList", "1", pars=params).json()["response"]

    appList.extend(requestAppList.get("apps", []))

    if not requestAppList.get("have_more_results", False):
        break

    if "last_appid" in requestAppList:
        json_params["last_appid"] = requestAppList["last_appid"]
        params["input_json"] = json.dumps(json_params)



Сохраняем данные в row

In [ ]:
with open("../data/row/main_games.json", 'w', encoding="utf-8") as f:
    json.dump(appList, f, indent=4, ensure_ascii=False)

with open("../data/row/main_games.json", 'r', encoding="utf-8") as f:
    print(json.load(f))


## Список